# Step 9: LightGBM Global Model Backtest

In this notebook, we train and evaluate a global LightGBM model on the tabular features engineered using Spark.

### Why one model per store instead of one model across all 30k series?
- 10 stores × ~940 days (d >= 1000) × ~30 features ≈ 30M rows, which risks running out of RAM in local pandas.
- A single store (`CA_1`) has ~2.8M rows. Training one model per store is still a true **global model**: it learns joint patterns across all 3,049 items in that store.
- Per-store and per-department models were among the top-performing strategies in the original M5 competition.

In [ ]:
import sys
from pathlib import Path

# Add project root to sys.path
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import lightgbm as lgb
import mlflow
import numpy as np
import pandas as pd

from src.backtest import split_fold
from src.config import CUTOFFS, DEV_FOLDS, EXPORTS, HOLDOUT, SEED, TRAIN_START_D
from src.metrics import coverage, eligible_ids, pinball, wape, wrmsse

STORES = ["CA_1", "CA_2", "CA_3", "CA_4", "TX_1", "TX_2", "TX_3", "WI_1", "WI_2", "WI_3"]
CATS = ["item_id", "dept_id", "cat_id", "event_type_1"]
FEATURES = ([
    "lag_28", "lag_35", "lag_42", "lag_364",
    "rmean_7", "rmean_28", "rmean_56", "rstd_28",
    "dow", "dom", "month", "snap", "is_event",
    "sell_price", "price_norm", "price_change_7d", "price_rel_dept"
] + CATS)

print("Configuration and modules loaded successfully.")
print(f"Selected {len(FEATURES)} features: {FEATURES}")

In [ ]:
# Load scale and weight metadata for scoring
stats = pd.read_parquet(EXPORTS / "series_stats")
print(f"Loaded series stats with shape: {stats.shape}")

def load_store(store: str) -> pd.DataFrame:
    """Loads the feature table for a single store and optimizes memory usage.
    
    - Converts categorical columns to 'category' before train/test split to freeze category codes.
    - Casts float64 numerical columns to float32 to reduce memory footprint by ~50%.
    """
    df = pd.read_parquet(EXPORTS / "features" / f"store_id={store}")
    for c in CATS:
        df[c] = df[c].astype("category")  # Category codes fixed before train/test split
    num = df.select_dtypes("float64").columns
    df[num] = df[num].astype("float32")  # Downcast float64 to float32
    return df

# Test loading store CA_1
df_ca1 = load_store("CA_1")
print(f"CA_1 loaded successfully: {len(df_ca1):,} rows, {df_ca1.shape[1]} columns")
print(f"Memory usage: {df_ca1.memory_usage().sum() / 1e6:.1f} MB")

### 9.3 Train, Early-Stop Correctly, Refit

#### Why Tweedie Loss (`tweedie_variance_power=1.2`)?
- Daily unit sales are non-negative counts with an excess of zeros (intermittent demand) and a skewed positive distribution.
- Standard MSE assumes Gaussian residuals and can produce negative sales predictions.
- Tweedie with $1 < p < 2$ (Compound Poisson-Gamma) directly models zero-inflated continuous/count distributions and guarantees non-negative predictions.

#### Leakage-Free Early Stopping
- **Critical rule:** Early stopping must **never** be performed on the 28-day test window (that would leak future information and produce over-optimistic evaluation scores).
- **Solution:** We split the training data internally into `inner` and `es` (the last 28 days of the training period). Once optimal iterations (`best_iteration`) are found, the model is refit on the entire training set (`train`).

In [ ]:
PARAMS = dict(
    objective="tweedie",
    tweedie_variance_power=1.2,
    learning_rate=0.05,
    num_leaves=127,
    min_data_in_leaf=100,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=1,
    lambda_l2=0.1,
    metric="rmse",
    verbose=-1,
    seed=SEED,
)


def fit_predict(df: pd.DataFrame, cutoff: int, params: dict, objective_override: dict | None = None):
    """Fits LightGBM using leakage-free early stopping and predicts on the holdout/test fold.
    
    1. Splits df into train (d <= cutoff) and test (cutoff < d <= cutoff + 28).
    2. Performs early stopping using the last 28 days of training (cutoff - 28 to cutoff).
    3. Refits on the entire training set with the discovered best_iteration.
    4. Predicts on test and returns (fitted_model, test_with_predictions).
    """
    p = {**params, **(objective_override or {})}
    train, test = split_fold(df, cutoff, TRAIN_START_D)

    # Early stopping on the LAST 28 DAYS OF TRAINING — never on the test window
    inner, es = split_fold(train, cutoff - 28, TRAIN_START_D)
    m = lgb.train(
        p,
        lgb.Dataset(inner[FEATURES], inner["sales"]),
        num_boost_round=3000,
        valid_sets=[lgb.Dataset(es[FEATURES], es["sales"])],
        callbacks=[lgb.early_stopping(100, verbose=False)],
    )

    # Refit on ALL training days with the chosen number of rounds
    final = lgb.train(
        p,
        lgb.Dataset(train[FEATURES], train["sales"]),
        num_boost_round=m.best_iteration,
    )
    return final, test.assign(yhat=final.predict(test[FEATURES]))

print("Step 9.3 setup complete: PARAMS defined and fit_predict function ready.")

### 9.4 The Backtest Loop, Logged to MLflow

Here we implement the standardized scoring and logging infrastructure:
1. **`log_eval` (THE Scorer):** Enforces identical evaluation criteria across all models (LightGBM and baselines), filters by `eligible_ids`, calculates WRMSSE, unit WAPE, dollar WAPE, and per-store WRMSSE, then exports predictions and logs artifacts to MLflow.
2. **`run_backtest`:** Orchestrates training and prediction across stores and folds, aggregating results and tagging runs.
3. **Baseline Logging:** Ingests baseline predictions generated in Step 5 and logs them through the exact same `log_eval` pipeline for unbiased head-to-head comparison.

In [ ]:
# Set tracking URI to local SQLite database
mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("m5-forecasting")


def log_eval(model_name: str, fold: str, pred: pd.DataFrame, params: dict, tags: dict | None = None):
    """THE scorer. pred columns: id, store_id, d_num, y, yhat, sell_price.
    Every model (baselines and ML) goes through this function to ensure uniform comparison.
    """
    st = stats[stats.fold == fold]
    path = EXPORTS / f"pred_{model_name}_{fold}.parquet"
    pred.to_parquet(path)  # all series kept
    scored = pred[pred["id"].isin(eligible_ids(st))]  # same series for every model

    with mlflow.start_run(run_name=f"{model_name}-{fold}"):
        mlflow.set_tags({"model": model_name, "fold": fold, **(tags or {})})
        mlflow.log_params(params)
        mlflow.log_metric("n_series_scored", scored["id"].nunique())
        mlflow.log_metric("wrmsse", wrmsse(scored, st))
        mlflow.log_metric("wape_units", wape(scored.y, scored.yhat))
        mlflow.log_metric("wape_dollars", wape(scored.y, scored.yhat, scored.sell_price))
        for s, g in scored.groupby("store_id"):
            mlflow.log_metric(f"wrmsse_{s}", wrmsse(g, st))
        mlflow.log_artifact(str(path))


def run_backtest(model_name: str, params: dict, folds=DEV_FOLDS, stores=STORES, tags: dict | None = None):
    """Runs the backtest loop across specified folds and stores, logging each fold to MLflow."""
    for fold in folds:
        preds = []
        for store in stores:
            df = load_store(store)
            _, p = fit_predict(df, CUTOFFS[fold], params)
            preds.append(p[["id", "d_num", "sales", "sell_price", "yhat"]].assign(store_id=store))
            del df
        pred = pd.concat(preds).rename(columns={"sales": "y"})
        log_eval(model_name, fold, pred, params, {"stores": ",".join(stores), **(tags or {})})


print("Step 9.4 logging and backtest functions defined.")

In [ ]:
# Log baselines through THE scorer so all models exist in the same MLflow table
actuals = pd.concat([
    pd.read_parquet(EXPORTS / "features" / f"store_id={s}", columns=["id", "d_num", "sales", "sell_price"]).assign(store_id=s)
    for s in STORES
]).rename(columns={"sales": "y"})

baselines = pd.read_parquet(EXPORTS / "baselines")

for fold in DEV_FOLDS:
    c = CUTOFFS[fold]
    act = actuals[(actuals.d_num > c) & (actuals.d_num <= c + 28)]
    for model_name, g in baselines[baselines.fold == fold].groupby("model"):
        pred = act.merge(g[["id", "d_num", "yhat"]], on=["id", "d_num"], how="inner")
        log_eval(model_name, fold, pred, params={"family": "statsforecast"}, tags={"stores": ",".join(STORES)})

print("Baselines successfully scored and logged to MLflow.")

### 9.5 Question 5 — LightGBM Baseline Backtest (lgbm_v1)

Train and evaluate default LightGBM (`lgbm_v1`) across validation folds F1 and F2 to test whether it genuinely beats classical time-series baselines.

In [ ]:
# Run default LightGBM model backtest across F1 and F2
run_backtest("lgbm_v1", PARAMS)
print("lgbm_v1 backtest completed and logged to MLflow.")

### 9.6 Question 6 — Tuning Done Right with Optuna

#### Best Practices Applied:
1. **Leak-free Tuning Split:** We tune strictly on the `TUNE` fold (cutoff day 1829, evaluation window 1830–1857). The tuned model is evaluated on F1 and F2 (subsequent windows it never saw during tuning).
2. **Item-Level Sampling (Not Row-Level):** We sample 500 unique items (`item_id`) across their entire time series for `CA_1`. This preserves uninterrupted temporal patterns and item hierarchy dynamics.
3. **TPE Bayesian Optimization:** 25 trials optimizing Tweedie objective parameters (`learning_rate`, `num_leaves`, `min_data_in_leaf`, `feature_fraction`, `lambda_l2`, `tweedie_variance_power`).
4. **Nested MLflow Logging:** Each trial is logged as a child run under the main study run.

In [ ]:
import optuna

# Prepare item-sampled tuning dataset from CA_1
rng = np.random.default_rng(SEED)
ca1 = load_store("CA_1")
sample_items = rng.choice(ca1["item_id"].cat.categories, size=500, replace=False)
tune_df = ca1[ca1["item_id"].isin(sample_items)].copy()  # Sample whole series, not random rows
tune_stats = stats[stats.fold == "TUNE"]


def objective(trial: optuna.Trial) -> float:
    params = {
        **PARAMS,
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.2, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 31, 255, log=True),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 20, 500, log=True),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.5, 1.0),
        "lambda_l2": trial.suggest_float("lambda_l2", 1e-3, 10.0, log=True),
        "tweedie_variance_power": trial.suggest_float("tweedie_variance_power", 1.05, 1.6),
    }
    _, p = fit_predict(tune_df, CUTOFFS["TUNE"], params)
    score = wrmsse(p.rename(columns={"sales": "y"}), tune_stats)

    with mlflow.start_run(run_name=f"optuna-{trial.number}", nested=True):
        mlflow.log_params(trial.params)
        mlflow.log_metric("wrmsse_tune", score)

    return score


# Execute Optuna study
with mlflow.start_run(run_name="optuna-study-CA_1-500items"):
    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=25)

print("Optimization finished!")
print("Best Hyperparameters:", study.best_params)
print(f"Best TUNE WRMSSE: {study.best_value:.5f}")

In [ ]:
# Run backtest with tuned hyperparameters across validation folds F1 and F2
BEST_PARAMS = {**PARAMS, **study.best_params}
run_backtest("lgbm_tuned", BEST_PARAMS)
print("lgbm_tuned backtest completed and logged to MLflow.")